In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

storage_account = "vesseldatavd99"
storage_key = dbutils.secrets.get(scope="vessel-secrets", key="storage-key")

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
spark.conf.set("spark.sql.shuffle.partitions", "64")

SILVER = f"abfss://silver@{storage_account}.dfs.core.windows.net/ais_clean"
GOLD = f"abfss://gold@{storage_account}.dfs.core.windows.net"

silver = spark.read.format("delta").load(SILVER)
print("silver rows:", silver.count())

silver rows: 287514155


In [0]:
valid = (silver
    .filter(F.col("validation_status") == "VALID")
    .filter(~F.col("key_conflict")))
print("valid, non-conflicting rows:", valid.count())

valid, non-conflicting rows: 287507025


In [0]:
R_KM = 6371.0
w = Window.partitionBy("MMSI").orderBy("ts")

legs = (valid
    .select("MMSI", "ts", "source_file_date", "lat", "lon", "sog")
    .withColumn("prev_lat", F.lag("lat").over(w))
    .withColumn("prev_lon", F.lag("lon").over(w))
    .withColumn("prev_ts",  F.lag("ts").over(w))
    .withColumn("gap_h", (F.unix_timestamp("ts") - F.unix_timestamp("prev_ts")) / 3600))

dlat = F.radians(F.col("lat") - F.col("prev_lat"))
dlon = F.radians(F.col("lon") - F.col("prev_lon"))
a = (F.pow(F.sin(dlat / 2), 2)
     + F.cos(F.radians("prev_lat")) * F.cos(F.radians("lat")) * F.pow(F.sin(dlon / 2), 2))
legs = legs.withColumn("km", 2 * R_KM * F.asin(F.sqrt(a)))
legs = legs.withColumn("kmh", F.col("km") / F.col("gap_h"))

# Keep only plausible legs: positive gap, at most 6h, and under 100 km/h
plausible = legs.filter(
    (F.col("gap_h") > 0) & (F.col("gap_h") <= 6) & (F.col("kmh") <= 100))

print("legs total:", legs.filter(F.col("prev_ts").isNotNull()).count())
print("legs kept:", plausible.count())

vessel_daily = (plausible.groupBy("MMSI", "source_file_date")
    .agg(F.count("*").alias("positions"),
         F.avg("sog").alias("avg_sog"),
         F.max("sog").alias("max_sog"),
         F.sum("km").alias("distance_km")))

legs total: 287476429
legs kept: 287294039


In [0]:
daily_kpis = (silver.groupBy("source_file_date")
    .agg(F.count("*").alias("rows"),
         F.sum(F.when(F.col("validation_status") == "VALID", 1).otherwise(0)).alias("valid"),
         F.sum(F.when(F.col("validation_status") == "FLAGGED", 1).otherwise(0)).alias("flagged"),
         F.sum(F.when(F.col("validation_status") == "INVALID", 1).otherwise(0)).alias("invalid"),
         F.countDistinct("MMSI").alias("distinct_vessels"))
    .withColumn("pct_flagged", F.round(F.col("flagged") / F.col("rows") * 100, 3))
    .orderBy("source_file_date"))

In [0]:
w_latest = Window.partitionBy("MMSI").orderBy(F.col("ts").desc())

vessel_ref = (valid
    .filter(F.col("VesselName").isNotNull())
    .withColumn("rn", F.row_number().over(w_latest))
    .filter("rn = 1")
    .select("MMSI", "VesselName", "IMO", "CallSign", "VesselType", "Length", "Width"))

In [0]:
for name, df in [("vessel_daily", vessel_daily),
                 ("daily_kpis", daily_kpis),
                 ("vessel_ref", vessel_ref)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"default.gold_{name}")
    print("gold table written:", name, flush=True)

gold table written: vessel_daily
gold table written: daily_kpis
gold table written: vessel_ref


In [0]:
for name in ["vessel_daily", "daily_kpis", "vessel_ref"]:
    t = spark.read.format("delta").load(f"{GOLD}/{name}")
    print(f"{name}: {t.count():,} rows")

display(spark.read.format("delta").load(f"{GOLD}/daily_kpis"))

vessel_daily: 570,666 rows
daily_kpis: 43 rows
vessel_ref: 27,424 rows


source_file_date,rows,valid,flagged,invalid,distinct_vessels,pct_flagged
2019-01-01,7515893,7515821,72,0,13115,0.001
2019-01-02,7168210,7168080,130,0,13440,0.002
2019-01-03,6722484,6722352,132,0,13434,0.002
2019-01-04,6949442,6949337,105,0,13790,0.002
2019-01-05,6849562,6849316,246,0,13715,0.004
2019-01-06,6966697,6966434,263,0,13629,0.004
2019-01-07,7026341,7026222,119,0,13887,0.002
2019-01-08,7128221,7128040,181,0,13772,0.003
2019-01-09,7278489,7278167,322,0,12869,0.004
2019-01-10,7104040,7103967,73,0,12932,0.001


In [0]:
vd = spark.read.format("delta").load(f"{GOLD}/vessel_daily")

display(vd.agg(
    F.min("distance_km").alias("min_km"),
    F.avg("distance_km").alias("avg_km"),
    F.max("distance_km").alias("max_km"),
    F.percentile_approx("max_sog", 0.5).alias("median_max_sog"),
    F.sum(F.when(F.col("max_sog") > 50, 1).otherwise(0)).alias("days_max_sog_over_50")))

min_km,avg_km,max_km,median_max_sog,days_max_sog_over_50
0.0,51.39045913700479,1108.0422661777357,3.9,0
